In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print(df.shape)
print(df.head(2))

In [ ]:
print(df.dtypes.value_counts())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:")
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

df.drop('customerID', axis=1, inplace=True)

df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Cleaned shape:", df.shape)
print("Churn rate:", df['Churn'].mean().round(3))

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols.remove('Churn')
corr = df[num_cols + ['Churn']].corr()['Churn'].drop('Churn').sort_values()
print("Correlations with Churn:")
print(corr.round(3))

cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols[:4]:
    print(f"\n{col}:")
    print(df.groupby(col)['Churn'].mean().round(3).sort_values(ascending=False))


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print(classification_report(y_test, dummy.predict(X_test)))